**CI twin of `ch16-feature-engineering.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
import pandas as pd

pg = load_csv("penguins").dropna(subset=["body_mass_g", "sex"])
print(pd.crosstab(pg["island"], pg["species"]))

In [ ]:
demo = pd.DataFrame({"island": ["Torgersen", "Biscoe", "Dream"]})
print(pd.get_dummies(demo, dtype=int))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

y = pg["species"]
Xtr, Xte, ytr, yte = train_test_split(
    pg[["body_mass_g", "island", "sex"]], y,
    test_size=0.25, random_state=42, stratify=y)

mass_only = Pipeline([
    ("prep", ColumnTransformer([("num", StandardScaler(), ["body_mass_g"])])),
    ("model", LogisticRegression(max_iter=1000)),
]).fit(Xtr, ytr)

with_words = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), ["body_mass_g"]),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["island", "sex"]),
    ])),
    ("model", LogisticRegression(max_iter=1000)),
]).fit(Xtr, ytr)

print(f"mass only:            {accuracy_score(yte, mass_only.predict(Xte)):.3f}")
print(f"mass + island + sex:  {accuracy_score(yte, with_words.predict(Xte)):.3f}")

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

homes = load_csv("california-housing-sample")
feats = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
         "Population", "AveOccup", "Latitude", "Longitude"]
yh = homes["MedHouseVal"]

homes["BedroomShare"] = homes["AveBedrms"] / homes["AveRooms"]
homes["RoomsPerPerson"] = homes["AveRooms"] / homes["AveOccup"]

for cols, label in [(feats, "8 raw features     "),
                    (feats + ["BedroomShare", "RoomsPerPerson"],
                     "+ 2 engineered      ")]:
    s = -cross_val_score(LinearRegression(), homes[cols], yh,
                         cv=5, scoring="neg_mean_absolute_error")
    print(f"{label}: CV MAE {s.mean():.3f} ± {s.std():.3f}")

In [ ]:
pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression()),
])

# fit() fits the scaler on THESE rows, then the model on the scaled result.
# There is no code path by which test rows can reach the scaler's fit.
scores = -cross_val_score(pipe, homes[feats], yh,
                          cv=5, scoring="neg_mean_absolute_error")
print(f"pipeline CV MAE: {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

eng = feats + ["BedroomShare", "RoomsPerPerson"]
homes_tr, homes_te, price_tr, price_te = train_test_split(homes[eng], yh,
                                        test_size=0.2, random_state=42)

search = GridSearchCV(
    Pipeline([("scale", StandardScaler()), ("model", Ridge())]),
    {"model__alpha": [0.01, 0.1, 1, 10, 100]},
    cv=5, scoring="neg_mean_absolute_error")
search.fit(homes_tr, price_tr)

print(f"CV chose:        {search.best_params_}")
print(f"its CV MAE:      {-search.best_score_:.3f}")
print(f"sealed test MAE: {mean_absolute_error(price_te, search.predict(homes_te)):.3f}")

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pg = load_csv("penguins").dropna(
    subset=["bill_length_mm", "bill_depth_mm",
            "flipper_length_mm", "body_mass_g", "sex"])
num = ["bill_length_mm", "bill_depth_mm",
       "flipper_length_mm", "body_mass_g"]
y = pg["species"]

pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])
scores = cross_val_score(pipe, pg[num], y, cv=5)

run_tests([
    ("five folds, preprocessing re-fitted in each", len(scores), 5),
    ("cross-validated accuracy", round(float(scores.mean()), 3), 0.991),
])

In [ ]:
def one_hot(value, categories):
    return [1 if value == c else 0 for c in categories]

def ratio_feature(numerators, denominators):
    return [round(n / d, 4) for n, d in zip(numerators, denominators)]

islands = ["Biscoe", "Dream", "Torgersen"]

run_tests([
    ("a Dream penguin", one_hot("Dream", islands), [0, 1, 0]),
    ("a Torgersen penguin", one_hot("Torgersen", islands), [0, 0, 1]),
    ("an island never seen in training", one_hot("Deception", islands),
     [0, 0, 0]),
    ("bedroom share", ratio_feature([1.0, 2.0], [4.0, 8.0]),
     [0.25, 0.25]),
    ("rooms per person", ratio_feature([6.0, 5.0], [3.0, 2.0]),
     [2.0, 2.5]),
])